In [1]:
!pip install rdkit
!pip install mordred

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.3/34.3 MB 50.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 128.8/128.8 kB 3.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 32.5 MB/s eta 0:00:00
  Created wheel for mordred: filename=mordred-1.2.0-py3-none-any.whl size=176718 sha256=a98dc269306d402fea7c1cc25647b43833db52714062ddca85159dd1898076f3
  Stored in directory: /root/.cache/pip/wheels/a7/4f/b8/d4c6591f6ac944aaced7865b349477695f662388ad958743c7
Successfully built mordred
  Attempting uninstall: networkx
    Found existing installation: networkx 3.4.2
    Uninstalling networkx-3.4.2:
      Successfully uninstalled networkx-3.4.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
nx-cugraph-cu12 24.10.0 requires networkx>=3.0, but you have networkx 2.8.8 which is incompatible.


In [2]:
import os
import re
import random
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")
from tqdm import tqdm
tqdm.pandas()

# Set seed for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# RDKit and Mordred imports
from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors3D, Descriptors, rdMolDescriptors
from mordred import Calculator, descriptors


In [3]:
# Directory containing the sdf and pdb files
pdb_folder = "/kaggle/input/pdb-all/PDB_all"

# Dictionary to store mapping:
# key = file base name, value = full file path (with .sdf taking precedence over .pdb)
file_mapping = {}

# Loop through each file in the directory
for file_name in os.listdir(pdb_folder):
    if file_name.endswith(('.sdf', '.pdb')):
        base, ext = os.path.splitext(file_name)
        ext = ext.lower()  # Normalize extension to lowercase
        full_path = os.path.join(pdb_folder, file_name)
        
        # If the file is an SDF, assign or override regardless of previous file type.
        if ext == '.sdf':
            file_mapping[base] = full_path
        # If the file is a PDB, only assign if there isn't already an SDF stored.
        elif ext == '.pdb':
            if base not in file_mapping:
                file_mapping[base] = full_path

# Print the mapping (or process it as needed)
# print(file_mapping)

In [ ]:
# Function to extract candidate molecule IDs from a CSV row
def extract_ids(mol_str):
    # If the molecule string contains letters, assume it's a full name
    if re.search(r'[a-zA-Z]', mol_str):
        return [mol_str]
    else:
        # Else split by underscore, e.g. "2389_38229" -> ["2389", "38229"]
        tokens = [mol_str]#re.findall(r'\d+', mol_str)
        return tokens

In [ ]:
# Initialize Mordred Calculator with all descriptors (0D, 1D, 2D and 3D)

calc = Calculator(descriptors, ignore_3D=False)

# Function to compute descriptors for one molecule row using pre-existing 3D files

def compute_rdkit_descriptors(mol):
    rdkit_desc = {}
    # Compute all descriptors available from RDKit's Descriptors module.
    # (These include many 0D, 1D and 2D descriptors.)
    for name, func in Descriptors.descList:
        # Directly compute
        rdkit_desc[f"RDKit_{name}"] = func(mol)
        
    # Compute additional 3D descriptors from Descriptors3D.
    extra_desc = {
        'RDKit_PBF': Descriptors3D.PBF(mol),
        'RDKit_PMI1': Descriptors3D.PMI1(mol),
        'RDKit_PMI2': Descriptors3D.PMI2(mol),
        'RDKit_PMI3': Descriptors3D.PMI3(mol),
        'RDKit_NPR1': Descriptors3D.NPR1(mol),
        'RDKit_NPR2': Descriptors3D.NPR2(mol),
        'RDKit_RadiusOfGyration': Descriptors3D.RadiusOfGyration(mol),
        'RDKit_InertialShapeFactor': Descriptors3D.InertialShapeFactor(mol),
        'RDKit_Eccentricity': Descriptors3D.Eccentricity(mol),
        'RDKit_Asphericity': Descriptors3D.Asphericity(mol),
        'RDKit_SpherocityIndex': Descriptors3D.SpherocityIndex(mol),
        'RDKit_Volume': AllChem.ComputeMolVolume(mol)
    }
    rdkit_desc.update(extra_desc)
    return rdkit_desc

def compute_descriptors(row):
    mol_field = str(row['Molecule'])
    candidate_ids = extract_ids(mol_field)
    
    file_path = None
    # Look for any candidate ID in our mapping
    for cid in candidate_ids:
        if cid in file_mapping:
            file_path = file_mapping[cid]  # use the first matching file
            break
    
    if file_path is None:
        print(f"File not found for molecule id {mol_field}. Generating molecule from SMILES.")
        mol = Chem.MolFromSmiles(row['SMILES'])
        if mol is None:
            print(f"Failed to generate molecule from SMILES for molecule id {mol_field}")
            return {}  # Return empty dictionary instead of None
        # Prepare the molecule: add hydrogens, embed and optimize using MMFF94
        mol = Chem.AddHs(mol)
        AllChem.EmbedMolecule(mol, randomSeed=42)
        AllChem.MMFFOptimizeMolecule(mol)
    else:
        mol = None
        # Load molecule from file based on extension
        if file_path.endswith('.sdf'):
            suppl = Chem.SDMolSupplier(file_path)
            for m in suppl:
                if m is not None:
                    mol = m
                    break
        elif file_path.endswith('.pdb'):
            mol = Chem.MolFromPDBFile(file_path)
        else:
            print(f"Unsupported file format: {file_path}")
            return {}  # Return empty dictionary instead of None
            
        if mol is None:
            print(f"Could not read molecule from {file_path} for molecule id {mol_field}")
            return {}  # Return empty dictionary instead of None
        
    # # Calculate Mordred descriptors
    # try:
    #     desc = calc(mol)
    #     mordred_desc = {str(k): float(v) if v not in [None, "nan"] else np.nan for k, v in desc.items()}
    # except Exception as e:
    #     print(f"Mordred descriptor calculation failed for molecule id {mol_field}: {e}")
    #     mordred_desc = {}

    # Calculate RDKit descriptors (including additional 3D descriptors)
    try:
        rdkit_desc = compute_rdkit_descriptors(mol)
    except Exception as e:
        print(f"RDKit descriptor calculation failed for molecule id {mol_field}: {e}")
        rdkit_desc = {}

    # Merge both descriptor dictionaries and return
    all_desc = {**rdkit_desc} #**mordred_desc,
    return all_desc

In [ ]:

# Read CSV files into dataframes
train_file = "/kaggle/input/scaffold-split/scaffold_train_dataset.csv"
val_file   = "/kaggle/input/scaffold-split/scaffold_validation_dataset.csv"
test_file  = "/kaggle/input/scaffold-split/scaffold_test_dataset.csv"

train_df = pd.read_csv(train_file)
val_df   = pd.read_csv(val_file)
test_df  = pd.read_csv(test_file)

# Compute descriptors for each dataset
print("Calculating descriptors for training set...")
train_desc_series = train_df.progress_apply(compute_descriptors, axis=1)
train_desc_df = pd.DataFrame(list(train_desc_series))
print("Training descriptors shape:", train_desc_df.shape)

print("Calculating descriptors for validation set...")
val_desc_series = val_df.progress_apply(compute_descriptors, axis=1)
val_desc_df = pd.DataFrame(list(val_desc_series))
print("Validation descriptors shape:", val_desc_df.shape)

print("Calculating descriptors for test set...")
test_desc_series = test_df.progress_apply(compute_descriptors, axis=1)
test_desc_df = pd.DataFrame(list(test_desc_series))
print("Test descriptors shape:", test_desc_df.shape)

Calculating descriptors for training set...


 90%|█████████ | 1490/1650 [06:27<00:51,  3.10it/s]

File not found for molecule id 1092593. Generating molecule from SMILES.


 90%|█████████ | 1491/1650 [06:28<01:12,  2.19it/s]

File not found for molecule id 1092615. Generating molecule from SMILES.


 90%|█████████ | 1492/1650 [06:29<01:19,  1.98it/s]

... (265 outputs left out)

100%|██████████| 335/335 [01:51<00:00,  1.59it/s]

File not found for molecule id 902630. Generating molecule from SMILES.


100%|██████████| 335/335 [01:51<00:00,  3.00it/s]

Test descriptors shape: (335, 229)


In [7]:
temp_train_df = pd.concat([train_df.reset_index(drop=True), train_desc_df], axis=1)
final_train_df = temp_train_df.drop(['Affinity1', 'Affinity2','Affinity3','Affinity4','Affinity5', 'Deviation', 'source', 'scaffold', 'split'], axis=1)
final_train_df.to_csv('scaffold_train_descriptors.csv', index=False)

temp_validation_df = pd.concat([val_df.reset_index(drop=True), val_desc_df], axis=1)
final_validation_df = temp_validation_df.drop(['Affinity1', 'Affinity2','Affinity3','Affinity4','Affinity5', 'Deviation', 'source', 'scaffold', 'split'], axis=1)
final_validation_df.to_csv('scaffold_validation_descriptors.csv', index=False)

temp_test_df = pd.concat([test_df.reset_index(drop=True), test_desc_df], axis=1)
final_test_df = temp_test_df.drop(['Affinity1', 'Affinity2','Affinity3','Affinity4','Affinity5', 'Deviation', 'source', 'scaffold', 'split'], axis=1)
final_test_df.to_csv('scaffold_test_descriptors.csv', index=False)